# Gold Layer - Datasets Analiticos

**Camada:** Gold (analitica)
**Entrada:** `workspace.silver.*`  |  **Saida:** `workspace.gold.*`

Cria datasets prontos para dashboards, analises estatisticas e ML. Como a **integracao**
das bases ja foi feita na Silver (tabelas `indicador_*_integrado`, com metas e gap), aqui o
foco e agregacao, ranking, evolucao temporal e features.

Datasets gerados:
1. `indicadores_municipio` - indicador por municipio com classificacoes
2. `metas_vs_resultados_municipio` - meta x resultado por municipio (gap, alcance, status)
3. `metas_vs_resultados_uf` - meta x resultado por UF
4. `evolucao_temporal_municipio` - variacao ano a ano (lag) e tendencia
5. `agregacoes_uf` - consolidado por UF com ranking nacional
6. `metricas_brasil` - KPIs nacionais + metas Brasil
7. `features_ml` - features prontas para modelagem

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("tech-challenge-gold").getOrCreate()

SILVER = "workspace.silver"
GOLD_CATALOG = "workspace"
GOLD_SCHEMA = "gold"
GOLD = f"{GOLD_CATALOG}.{GOLD_SCHEMA}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")
print(f"Schema {GOLD} pronto")

def salvar_gold(df, tabela):
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{GOLD}.{tabela}")
    print(f"{tabela}: {df.count():,} registros")

In [0]:
# Tabelas Silver (incluindo as integradas indicador x meta)
indicador_municipio = spark.table(f"{SILVER}.indicador_alfabetizacao_municipio")
indicador_uf = spark.table(f"{SILVER}.indicador_alfabetizacao_uf")
meta_brasil = spark.table(f"{SILVER}.meta_alfabetizacao_brasil")
integ_municipio = spark.table(f"{SILVER}.indicador_municipio_integrado")
integ_uf = spark.table(f"{SILVER}.indicador_uf_integrado")
print("Silver carregada")

## 1. Indicadores por municipio (com classificacoes)

In [0]:
indicadores_municipio = (
    integ_municipio
    .withColumn("nivel_alfabetizacao",
        F.when(F.col("taxa_alfabetizacao") >= 80, "Alto")
         .when(F.col("taxa_alfabetizacao") >= 60, "Medio")
         .when(F.col("taxa_alfabetizacao") >= 40, "Baixo")
         .otherwise("Muito Baixo"))
    .withColumn("faixa_proficiencia",
        F.when(F.col("media_portugues") >= 800, "Avancado")
         .when(F.col("media_portugues") >= 750, "Adequado")
         .when(F.col("media_portugues") >= 700, "Basico")
         .otherwise("Abaixo do Basico"))
)
salvar_gold(indicadores_municipio, "indicadores_municipio")

## 2 e 3. Metas x resultados (municipio e UF)

In [0]:
_metas_expr = ("stack(7, '2024', meta_2024, '2025', meta_2025, '2026', meta_2026, "
               "'2027', meta_2027, '2028', meta_2028, '2029', meta_2029, "
               "'2030', meta_2030) as (ano_meta, meta)")

def metas_vs_resultados(integ, chaves):
    base = integ.select(*chaves, "taxa_alfabetizacao", F.expr(_metas_expr))
    return (base
        .withColumn("ano_meta", F.col("ano_meta").cast("int"))
        .filter(F.col("meta").isNotNull())
        .withColumn("gap_meta", F.col("meta") - F.col("taxa_alfabetizacao"))
        .withColumn("percentual_alcance",
                    F.when(F.col("meta") != 0, (F.col("taxa_alfabetizacao") / F.col("meta")) * 100))
        .withColumn("status_meta",
            F.when(F.col("taxa_alfabetizacao") >= F.col("meta"), "Atingida")
             .when(F.col("percentual_alcance") >= 90, "Proximo")
             .when(F.col("percentual_alcance") >= 75, "Em Progresso")
             .otherwise("Abaixo do Esperado")))

metas_vs_resultados_municipio = metas_vs_resultados(integ_municipio, ["ano", "id_municipio", "codigo_uf", "rede"])
metas_vs_resultados_uf = metas_vs_resultados(integ_uf, ["ano", "sigla_uf", "rede"])
salvar_gold(metas_vs_resultados_municipio, "metas_vs_resultados_municipio")
salvar_gold(metas_vs_resultados_uf, "metas_vs_resultados_uf")

## 4. Evolucao temporal por municipio

In [0]:
w_muni = Window.partitionBy("id_municipio", "rede").orderBy("ano")

evolucao_temporal_municipio = (
    indicador_municipio
    .withColumn("taxa_ano_anterior", F.lag("taxa_alfabetizacao").over(w_muni))
    .withColumn("variacao_absoluta", F.col("taxa_alfabetizacao") - F.col("taxa_ano_anterior"))
    .withColumn("variacao_percentual",
        F.when(F.col("taxa_ano_anterior").isNotNull() & (F.col("taxa_ano_anterior") != 0),
               ((F.col("taxa_alfabetizacao") - F.col("taxa_ano_anterior")) / F.col("taxa_ano_anterior")) * 100))
    .withColumn("tendencia",
        F.when(F.col("variacao_absoluta") > 2, "Crescimento Forte")
         .when(F.col("variacao_absoluta") > 0, "Crescimento")
         .when(F.col("variacao_absoluta") < -2, "Queda Forte")
         .when(F.col("variacao_absoluta") < 0, "Queda")
         .otherwise("Estavel"))
    .withColumn("codigo_uf", F.substring(F.col("id_municipio"), 1, 2))
)
salvar_gold(evolucao_temporal_municipio, "evolucao_temporal_municipio")

## 5. Agregacoes por UF (com ranking nacional)

In [0]:
w_rank = Window.partitionBy("ano", "rede").orderBy(F.desc("taxa_alfabetizacao_media"))

agregacoes_uf = (
    indicador_uf
    .groupBy("ano", "sigla_uf", "rede")
    .agg(
        F.avg("taxa_alfabetizacao").alias("taxa_alfabetizacao_media"),
        F.max("taxa_alfabetizacao").alias("taxa_alfabetizacao_max"),
        F.min("taxa_alfabetizacao").alias("taxa_alfabetizacao_min"),
        F.stddev("taxa_alfabetizacao").alias("taxa_alfabetizacao_desvio"),
        F.avg("media_portugues").alias("media_portugues_media"),
        F.count("*").alias("num_registros"))
    .withColumn("ranking_nacional", F.row_number().over(w_rank))
    .withColumn("classificacao",
        F.when(F.col("ranking_nacional") <= 5, "Top 5")
         .when(F.col("ranking_nacional") <= 10, "Top 10")
         .otherwise("Demais"))
)
salvar_gold(agregacoes_uf, "agregacoes_uf")

## 6. Metricas nacionais (Brasil)

In [0]:
metricas_brasil = (
    indicador_municipio
    .groupBy("ano", "rede")
    .agg(
        F.avg("taxa_alfabetizacao").alias("taxa_alfabetizacao_media_brasil"),
        F.stddev("taxa_alfabetizacao").alias("taxa_alfabetizacao_desvio_brasil"),
        F.avg("media_portugues").alias("media_portugues_brasil"),
        F.countDistinct("id_municipio").alias("num_municipios"))
    .join(
        meta_brasil.select("ano", "rede", "meta_2024", "meta_2030"),
        on=["ano", "rede"], how="left")
)
salvar_gold(metricas_brasil, "metricas_brasil")

## 7. Features para Machine Learning

Usa a distribuicao por nivel oriunda do INEP (`nivel_0..8`, trazida para a tabela integrada
na Silver). `soma_niveis_basicos` = niveis 0-2 (alunos em estagios iniciais) e
`soma_niveis_avancados` = niveis 6-8 (alunos proficientes) resumem o formato da distribuicao.

In [0]:
BASICOS = ["nivel_0", "nivel_1", "nivel_2"]
AVANCADOS = ["nivel_6", "nivel_7", "nivel_8"]

features_ml = (
    integ_municipio
    .withColumn("soma_niveis_basicos", sum(F.coalesce(F.col(c), F.lit(0.0)) for c in BASICOS))
    .withColumn("soma_niveis_avancados", sum(F.coalesce(F.col(c), F.lit(0.0)) for c in AVANCADOS))
    .select(
        "ano", "id_municipio", "codigo_uf", "rede",
        "taxa_alfabetizacao", "media_portugues",
        "meta_2030", "gap_meta_2030",
        "soma_niveis_basicos", "soma_niveis_avancados",
        *[f"nivel_{i}" for i in range(9)])
)
salvar_gold(features_ml, "features_ml")

## Validacao final

In [0]:
tabelas = spark.sql(f"SHOW TABLES IN {GOLD}").collect()
print(f"Tabelas gold criadas ({len(tabelas)}):")
for t in tabelas:
    c = spark.table(f"{GOLD}.{t.tableName}").count()
    print(f"  {t.tableName}: {c:,} registros")

## Exportacao para S3 (backup Parquet)

Exporta copias das Delta Tables para S3 em formato Parquet, organizadas por camada.

In [0]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "boto3", "python-dotenv", "pyarrow"])

from io import BytesIO
from pathlib import Path
import boto3
from dotenv import load_dotenv
import pyarrow as pa
import pyarrow.parquet as pq

# Configuracao
S3_BUCKET = "amzn-s3-fiap-tech2"
S3_PREFIX = "processed/"
CATALOG = "workspace"
CAMADAS = ["bronze", "silver", "gold"]
ENV_PATH = Path("/Workspace/Users/justinofilipe03@gmail.com/TechChallenge_2/.env")

load_dotenv(ENV_PATH)
s3 = boto3.client("s3")

print(f"Exportando para s3://{S3_BUCKET}/{S3_PREFIX}\n")

totais = {}
for camada in CAMADAS:
    print(f"[{camada.upper()}]")
    full_schema = f"{CATALOG}.{camada}"
    tabelas = spark.sql(f"SHOW TABLES IN {full_schema}").collect()
    
    total_registros = 0
    for t in tabelas:
        tabela = t.tableName
        df_spark = spark.table(f"{full_schema}.{tabela}")
        count = df_spark.count()
        
        # Converte Spark -> Pandas -> Arrow -> Parquet em memoria
        df_pandas = df_spark.toPandas()
        arrow_table = pa.Table.from_pandas(df_pandas)
        
        buffer = BytesIO()
        pq.write_table(arrow_table, buffer)
        buffer.seek(0)
        
        # Upload para S3
        s3_key = f"{S3_PREFIX}{camada}/{tabela}.parquet"
        s3.put_object(Bucket=S3_BUCKET, Key=s3_key, Body=buffer.getvalue())
        
        total_registros += count
        print(f"  {tabela}: {count:,} registros")
    
    totais[camada] = total_registros
    print()

print("Exportacao concluida:")
for camada, total in totais.items():
    print(f"  {camada}: {total:,} registros")

# Verifica arquivos criados
print(f"\nArquivos em s3://{S3_BUCKET}/{S3_PREFIX}:")
for camada in CAMADAS:
    response = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=f"{S3_PREFIX}{camada}/")
    if "Contents" in response:
        print(f"  {camada}: {len(response['Contents'])} arquivo(s)")